# Part 4 · Explore saved simulation results

Opens one or more results (`<RUN_NAME>_results.zip` or the run folder) **without calling the API and without
re-running the simulation**: interactive replay, headline numbers, cell populations over time, events,
decisions per cell type, the decision history of a single cell, and a comparison table across runs.

Runs on any Colab runtime (CPU) or locally. Guide: [docs/04_part4_results.md](https://github.com/hylixinsights/VirtualTissue-Xenium/blob/main/docs/04_part4_results.md).
Tip: `replay.html` inside the ZIP also opens directly in Chrome/Firefox/Safari, with no notebook at all.


In [ ]:
#@title Get the VirtualTissue-Xenium code and install the pinned dependencies
# ✏️ EDIT only if needed — where the code comes from:
#   github        clone the public repository (default)
#   google_drive  copy the VirtualTissue-Xenium folder you placed in Google Drive
#   installed     the package is already installed with pip (local Jupyter)
CODE_SOURCE = "github"  #@param ["github", "google_drive", "installed"]
GITHUB_URL = "https://github.com/hylixinsights/VirtualTissue-Xenium.git"  #@param {type:"string"}
GITHUB_BRANCH = "main"  #@param {type:"string"}
DRIVE_CODE_DIR = "/content/drive/MyDrive/VirtualTissue-Xenium"  #@param {type:"string"}


import importlib, os, shutil, subprocess, sys
from pathlib import Path
print("Running Python:", sys.version.split()[0], "| executable:", sys.executable)
IN_COLAB = "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ
# Temporary Colab disk. Keep the code here, not in Drive.
CODE_DIR = Path("/content/VirtualTissue-Xenium")
if not IN_COLAB and CODE_SOURCE != "installed":
    # Local Jupyter: use the repository that contains this notebook.
    here = Path.cwd().resolve()
    CODE_DIR = here.parent if (here.parent / "src" / "virtualtissue_xenium").is_dir() else here
    CODE_SOURCE = "local_repository"
if CODE_SOURCE == "github":
    if not (CODE_DIR / "src" / "virtualtissue_xenium").is_dir():
        subprocess.check_call(["git", "clone", "--quiet", "--depth", "1", "--branch", GITHUB_BRANCH,
                               GITHUB_URL, str(CODE_DIR)])
elif CODE_SOURCE == "google_drive":
    from google.colab import drive
    drive.mount("/content/drive")
    if not (Path(DRIVE_CODE_DIR) / "src" / "virtualtissue_xenium").is_dir():
        raise FileNotFoundError("DRIVE_CODE_DIR must be the VirtualTissue-Xenium folder "
                                "(the one that contains src/, notebooks/ and README.md).")
    shutil.rmtree(CODE_DIR, ignore_errors=True)
    shutil.copytree(DRIVE_CODE_DIR, CODE_DIR,
                    ignore=shutil.ignore_patterns(".git", "__pycache__", "*.pyc", ".ipynb_checkpoints"))
if CODE_SOURCE != "installed" and str(CODE_DIR / "src") not in sys.path:
    sys.path.insert(0, str(CODE_DIR / "src"))
import virtualtissue_xenium as vtx
from virtualtissue_xenium import colab as vtx_colab
print("VirtualTissue-Xenium", vtx.__version__, "| engine:", vtx.ENGINE_DIR)
os.environ.setdefault("MPLCONFIGDIR", "/tmp/vtx_matplotlib")


# This notebook only reads saved results: pandas and matplotlib from the runtime are enough.
SETUP_READY = True

if not SETUP_READY:
    raise RuntimeError("Restart the session and rerun this setup cell before continuing.")
from virtualtissue_xenium import results as vtx_results
print("Ready.")


## ✏️ Which result to open
`RESULTS_PATH` can be a results ZIP or a run folder, in Drive (`/content/drive/MyDrive/...`) or on the session
disk. With `upload`, choose the ZIP from your computer.

**Local Jupyter:** leave `RESULTS_SOURCE = google_drive` and replace `RESULTS_PATH` with a local path;
this option reads local files without mounting Drive. For example, use
`vtx_session/results/R03_jev_run01_results.zip` when the simulation ran from the same notebook folder.
A local preview has the suffix `_local_preview_results.zip`. The `upload` chooser requires Colab.


In [ ]:
#@title ✏️ Result to open
RESULTS_SOURCE = "google_drive"  #@param ["google_drive", "upload"]
RESULTS_PATH = "/content/drive/MyDrive/VirtualTissue/Simulation_Results/R03_jev_run01_results.zip"  #@param {type:"string"}
import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
WORK = Path("/content/vtx_explore") if IN_COLAB else Path("vtx_explore").resolve()
WORK.mkdir(parents=True, exist_ok=True)
results_path = Path(RESULTS_PATH)
if RESULTS_SOURCE == "upload":
    from google.colab import files
    uploaded = files.upload()
    name, content = next(iter(uploaded.items()))
    results_path = WORK / Path(name).name
    results_path.write_bytes(content)
elif RESULTS_PATH.startswith("/content/drive/") and IN_COLAB:
    vtx_colab.mount_drive()
extract_to = WORK / results_path.stem
shutil.rmtree(extract_to, ignore_errors=True)
RES = vtx_results.load_results(results_path, extract_to=extract_to)
display(pd.Series(vtx_results.summary(RES), name="value").to_frame())


In [ ]:
#@title Interactive replay
vtx_colab.show_replay(RES["replay"])


In [ ]:
#@title Overview: populations, decisions, events, choices
fig = vtx_results.plot_overview(RES)
plt.show()
display(vtx_results.population_by_type(RES).iloc[[0, -1]])
display(vtx_results.choice_counts(RES))
display(vtx_results.event_counts(RES, bin_hours=6))


## ✏️ Follow one cell
Find a cell id in the replay's inspector (click a cell) or in the decisions table, then set `CELL_ID`.
Each row is one decision: the chosen action, Jev's probability for every permitted option, and why the
cell was asked (`wake_reasons`).


In [ ]:
#@title ✏️ Decision history of one cell
DECISIONS = vtx_results.decision_table(RES)
CELL_ID = 0  #@param {type:"integer"}
cell = CELL_ID or (int(DECISIONS.cell_id.iloc[0]) if not DECISIONS.empty else 0)
print("Cell", cell, "(type:", vtx_results.labels_by_body(RES).get(cell), ")")
pd.set_option("display.max_colwidth", 200)
display(vtx_results.cell_history(RES, cell))


## ✏️ Compare several runs (optional)
List result ZIPs or folders, separated by commas. Use the same region, seed, provider,
model and settings under `declared_recognition` and `recognition_off`, or identical
scenarios for a repeat check. Results are compared at the latest common saved time.
Local preview and Jev runs are different providers and cannot form a matched pair.


In [ ]:
#@title ✏️ Comparison table
COMPARE_PATHS = ""  #@param {type:"string"}
paths = [p.strip() for p in COMPARE_PATHS.replace(",", "\n").splitlines() if p.strip()]
if paths:
    try:
        display(vtx_results.compare_runs(paths))
    except ValueError as error:
        print("Comparison unavailable:", error)
else:
    print("Add paths to COMPARE_PATHS (comma-separated) to compare runs.")
